# 053 — CNN y aprendizaje espacial

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** (a) (28−3+2)/1+1 = 28 → **28×28×8** (padding 1 conserva el tamaño
con kernel 3). (b) (64−5)/2+1 = 30 (⌊59/2⌋+1) → **30×30×16**. (c) 28/2 = **14×14×8**.

**Ejercicio 2.** (a) 8·(3·3·1+1) = **80**. (b) 16·(5·5·3+1) = **1216**. Densa
equivalente de (b): (64·64·3)·(30·30·16) = 12288·14400 ≈ **1.77×10⁸** pesos:
la convolución usa ~145 000 veces menos.

**Ejercicio 3.** Cada salida = I(i,j)·1 − I(i,j+1) + 2·I(i+1,j+1):
S(1,1) = 1−0+2·1 = **3**; S(1,2) = 0−2+2·0 = **−2**;
S(2,1) = 2−1+2·3 = **7**; S(2,2) = 1−0+2·1 = **3**. S = [[3,−2],[7,3]].

**Ejercicio 4.** Stride 1: RF = 1+2+2+2 = **7×7** (cada 3×3 suma K−1=2). Con stride 2
en la segunda capa, la tercera suma (K−1)·2: RF = 1+2+2+4 = **9×9** — los strides
multiplican el crecimiento del campo receptivo de las capas posteriores.


In [ ]:
result = run_lab("perception", seed=53)
assert result["kind"] == "perception"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
def out_size(n, k, p, s):
    return (n - k + 2 * p) // s + 1

print("1a:", out_size(28, 3, 1, 1), "| 1b:", out_size(64, 5, 0, 2), "| 1c:", 28 // 2)
assert out_size(28, 3, 1, 1) == 28 and out_size(64, 5, 0, 2) == 30

print("params 1a:", 8 * (3*3*1 + 1), "| params 1b:", 16 * (5*5*3 + 1))

# Ejercicio 3: correlación 2D
I = [[1, 0, 2], [2, 1, 0], [0, 3, 1]]
K = [[1, -1], [0, 2]]
S = [[sum(I[i+m][j+n] * K[m][n] for m in range(2) for n in range(2))
      for j in range(2)] for i in range(2)]
print("S =", S)
assert S == [[3, -2], [7, 3]]

# Ejercicio 4: campo receptivo acumulado
rf, jump = 1, 1
for k, s in [(3, 1), (3, 1), (3, 1)]:
    rf += (k - 1) * jump
    jump *= s
print("RF tres 3×3 stride 1:", rf)


## Reflexión

1. ¿Por qué dos capas 3×3 apiladas son preferibles a una 7×7 en parámetros y en no linealidad, y qué pierden a cambio?
2. Un bloque residual aprende F(x) = H(x) − x. ¿Por qué "aprender el residuo" es más fácil cuando la transformación óptima está cerca de la identidad?
3. ¿Qué evidencia distinguiría un fallo por campo receptivo insuficiente de un fallo por falta de datos de una clase?
